# Experiment 5 — CNN (LeNet & Custom Architecture) on CIFAR-10

**Module 4 · Topic: Neural Networks · CO3**

### Problem Statement
> Design a CNN (e.g., **LeNet** or a custom architecture) to classify images from the
> **CIFAR-10** dataset.

### What this notebook does
1. Loads and explores CIFAR-10 (50 000 train / 10 000 test, 10 classes, 32x32 RGB).
2. Implements **LeNet-5** adapted to 3-channel 32x32 input — the historical baseline.
3. Implements a **custom VGG-style CNN** with batch-norm, dropout and global average pooling.
4. Implements a compact **ResNet-style CNN** with residual blocks.
5. Trains all three with identical budgets and compares them fairly.
6. Full evaluation: confusion matrix, per-class metrics, misclassification gallery, feature-map
   and filter visualisation, t-SNE of the learned embedding.

### How to run
`Runtime -> Change runtime type -> Hardware accelerator: **T4 GPU**`, then `Runtime -> Run all`.
Total runtime ~12–18 minutes on a T4.

## 0. Setup

In [ ]:
import os, time, random, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: enable the T4 GPU via Runtime -> Change runtime type")

## 1. Dataset — CIFAR-10

10 mutually exclusive classes, 6 000 images each. The images are tiny (32x32) and low-contrast,
so the hard pairs are **cat vs dog** and **automobile vs truck** — watch for those in the
confusion matrix at the end.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode="reflect"),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_full = datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
test_ds    = datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

n_val = 5000
train_ds, val_ds = random_split(train_full, [len(train_full) - n_val, n_val],
                                generator=torch.Generator().manual_seed(SEED))
val_ds.dataset = datasets.CIFAR10("./data", train=True, download=False, transform=test_tf)

BATCH = 128
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=512,   shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=512,   shuffle=False, num_workers=2, pin_memory=True)
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")

In [ ]:
# Class distribution + sample grid
raw = datasets.CIFAR10("./data", train=True, download=False)
labels = np.array(raw.targets)
counts = np.bincount(labels)

fig = plt.figure(figsize=(13, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax1.bar(CLASSES, counts, color="#4c8fbd"); ax1.set_title("CIFAR-10 class balance")
ax1.tick_params(axis="x", rotation=60); ax1.grid(axis="y", alpha=.3)

ax2 = fig.add_subplot(1, 2, 2); ax2.axis("off")
grid = np.zeros((32 * 5, 32 * 10, 3), dtype=np.uint8)
for c in range(10):
    idx = np.where(labels == c)[0][:5]
    for r, i in enumerate(idx):
        grid[r*32:(r+1)*32, c*32:(c+1)*32] = np.array(raw.data[i])
ax2.imshow(grid); ax2.set_title("5 samples per class (columns = classes)")
plt.tight_layout(); plt.show()
print("class order:", CLASSES)

## 2. Model A — LeNet-5 (1998, adapted)

Yann LeCun's original design: `conv(6) -> pool -> conv(16) -> pool -> FC120 -> FC84 -> FC10`.
Adapted here for 3-channel 32x32 input; we keep `tanh` in one variant to stay faithful, but use
ReLU by default since it trains far better.

**Limitations to expect:** only ~62 K parameters, no batch-norm, no padding, and the two
fully-connected layers hold most of the capacity — it will plateau around 65–72 % on CIFAR-10.

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, n_classes=10, act="relu"):
        super().__init__()
        A = nn.ReLU if act == "relu" else nn.Tanh
        self.features = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5),   # 32 -> 28
            A(),
            nn.AvgPool2d(2),                  # 28 -> 14
            nn.Conv2d(6, 16, kernel_size=5),  # 14 -> 10
            A(),
            nn.AvgPool2d(2),                  # 10 -> 5
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120), A(),
            nn.Linear(120, 84),        A(),
            nn.Linear(84, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(LeNet5())
print("LeNet-5 params:", f"{count_params(LeNet5()):,}")

## 3. Model B — Custom VGG-style CNN

Design decisions and the reason for each:

| Decision | Reason |
|---|---|
| 3x3 convs only, stacked in pairs | two 3x3 layers see a 5x5 receptive field with fewer parameters and an extra non-linearity |
| Channel widths 64 -> 128 -> 256 | spatial resolution halves at each pool, so channels double to keep capacity roughly constant |
| BatchNorm after every conv | stabilises and accelerates training, allows a higher learning rate |
| Global Average Pooling instead of a big flatten | removes ~1 M FC parameters, acts as a structural regularizer |
| Dropout 0.05 -> 0.1 -> 0.2 (increasing with depth) | deeper features are more specialised and more prone to co-adaptation |

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, n_classes=10, width=64, p=0.3):
        super().__init__()
        def conv_bn(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                                 nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        w = width
        self.block1 = nn.Sequential(conv_bn(3, w),      conv_bn(w, w),      nn.MaxPool2d(2), nn.Dropout(p * 0.15))
        self.block2 = nn.Sequential(conv_bn(w, 2*w),    conv_bn(2*w, 2*w),  nn.MaxPool2d(2), nn.Dropout(p * 0.35))
        self.block3 = nn.Sequential(conv_bn(2*w, 4*w),  conv_bn(4*w, 4*w),  nn.MaxPool2d(2), nn.Dropout(p * 0.65))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(p), nn.Linear(4*w, n_classes))
    def forward(self, x):
        return self.head(self.block3(self.block2(self.block1(x))))

print("CustomCNN params:", f"{count_params(CustomCNN()):,}")

## 4. Model C — Compact ResNet

Residual blocks let gradients flow through the identity path, so depth stops hurting.
`y = F(x) + x` — the network only has to learn the *residual* correction.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(cout)
        self.short = nn.Sequential()
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride=stride, bias=False),
                                       nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        return F.relu(out + self.short(x))

class SmallResNet(nn.Module):
    def __init__(self, n_classes=10, w=64):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, w, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(w), nn.ReLU(inplace=True))
        self.l1 = nn.Sequential(ResBlock(w, w),         ResBlock(w, w))
        self.l2 = nn.Sequential(ResBlock(w, 2*w, 2),    ResBlock(2*w, 2*w))
        self.l3 = nn.Sequential(ResBlock(2*w, 4*w, 2),  ResBlock(4*w, 4*w))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.2), nn.Linear(4*w, n_classes))
    def forward(self, x):
        return self.head(self.l3(self.l2(self.l1(self.stem(x)))))

print("SmallResNet params:", f"{count_params(SmallResNet()):,}")

## 5. Shared training / evaluation utilities

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval(); correct = total = 0; loss_sum = 0.0
    preds, ys = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        with torch.autocast("cuda", enabled=(device.type == "cuda")):
            out = model(xb)
            if criterion is not None:
                loss_sum += criterion(out, yb).item() * yb.size(0)
        p = out.argmax(1)
        correct += (p == yb).sum().item(); total += yb.size(0)
        preds.append(p.cpu()); ys.append(yb.cpu())
    return correct / total, loss_sum / max(total, 1), torch.cat(preds), torch.cat(ys)


def train_model(model, epochs=15, lr=1e-3, weight_decay=5e-4, label_smoothing=0.05, tag="model"):
    model = model.to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                                steps_per_epoch=len(train_dl), pct_start=0.25)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_val, best_state = 0.0, None
    t_start = time.time()
    for ep in range(1, epochs + 1):
        model.train(); run = c = n = 0
        t0 = time.time()
        for xb, yb in train_dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=(device.type == "cuda")):
                out = model(xb); loss = crit(out, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update()
            hist["lr"].append(sched.get_last_lr()[0]); sched.step()
            run += loss.item() * yb.size(0)
            c += (out.argmax(1) == yb).sum().item(); n += yb.size(0)
        va, vl, _, _ = evaluate(model, val_dl, crit)
        hist["train_loss"].append(run / n); hist["train_acc"].append(c / n)
        hist["val_loss"].append(vl);        hist["val_acc"].append(va)
        print(f"[{tag}] ep {ep:02d}/{epochs} | train acc {c/n:.4f} loss {run/n:.4f} "
              f"| val acc {va:.4f} loss {vl:.4f} | {time.time()-t0:.1f}s")
        if va > best_val:
            best_val = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    print(f"[{tag}] best val {best_val:.4f} | total {time.time()-t_start:.0f}s")
    return model, hist

## 6. Train all three models

In [ ]:
EPOCHS = 15   # 15 epochs ~= 4-5 min per model on a T4. Raise to 40 for ~93% with the ResNet.

results, histories = {}, {}

torch.manual_seed(SEED)
lenet, h = train_model(LeNet5(), epochs=EPOCHS, lr=2e-3, weight_decay=1e-4, tag="LeNet-5")
histories["LeNet-5"] = h
results["LeNet-5"] = evaluate(lenet, test_dl)[0]
print("LeNet-5 test acc:", round(results["LeNet-5"], 4), "\n")

In [ ]:
torch.manual_seed(SEED)
custom, h = train_model(CustomCNN(), epochs=EPOCHS, lr=2e-3, weight_decay=5e-4, tag="CustomCNN")
histories["CustomCNN"] = h
results["CustomCNN"] = evaluate(custom, test_dl)[0]
print("CustomCNN test acc:", round(results["CustomCNN"], 4), "\n")

In [ ]:
torch.manual_seed(SEED)
resnet, h = train_model(SmallResNet(), epochs=EPOCHS, lr=1.5e-3, weight_decay=5e-4, tag="SmallResNet")
histories["SmallResNet"] = h
results["SmallResNet"] = evaluate(resnet, test_dl)[0]
print("SmallResNet test acc:", round(results["SmallResNet"], 4), "\n")

## 7. Comparison

In [ ]:
models_map = {"LeNet-5": lenet, "CustomCNN": custom, "SmallResNet": resnet}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for name, h in histories.items():
    axes[0].plot(h["val_acc"],   marker="o", ms=3, label=name)
    axes[1].plot(h["val_loss"],  marker="o", ms=3, label=name)
    axes[2].plot(np.array(h["train_acc"]) - np.array(h["val_acc"]), marker="o", ms=3, label=name)
axes[0].set_title("Validation accuracy"); axes[1].set_title("Validation loss")
axes[2].set_title("train_acc - val_acc (overfitting gap)"); axes[2].axhline(0, color="k", lw=.8)
for a in axes:
    a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"{'model':<14}{'params':>12}{'test acc':>11}")
print("-" * 37)
for name, m in models_map.items():
    print(f"{name:<14}{count_params(m):>12,}{results[name]:>11.4f}")

## 8. Detailed evaluation of the best model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

best_name = max(results, key=results.get)
best = models_map[best_name]
acc, _, preds, ys = evaluate(best, test_dl)
preds_np, ys_np = preds.numpy(), ys.numpy()

print(f"BEST MODEL: {best_name}   test accuracy {acc:.4f}   macro-F1 "
      f"{f1_score(ys_np, preds_np, average='macro'):.4f}\n")
print(classification_report(ys_np, preds_np, target_names=CLASSES, digits=3))

In [ ]:
cm = confusion_matrix(ys_np, preds_np)
cmn = cm / cm.sum(1, keepdims=True)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=60, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES)
for i in range(10):
    for j in range(10):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"Row-normalised confusion matrix — {best_name}")
fig.colorbar(im); plt.tight_layout(); plt.show()

off = cmn.copy(); np.fill_diagonal(off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1][:6], off.shape))[0]
print("Most confused class pairs:")
for i, j in pairs:
    print(f"  {CLASSES[i]:<11} predicted as {CLASSES[j]:<11} {off[i,j]:.1%}")

In [ ]:
# Gallery of misclassified test images
raw_test = datasets.CIFAR10("./data", train=False, download=False)
wrong = np.where(preds_np != ys_np)[0]
sel = wrong[:20]
fig, axes = plt.subplots(2, 10, figsize=(18, 4))
for ax, i in zip(axes.ravel(), sel):
    ax.imshow(raw_test.data[i]); ax.axis("off")
    ax.set_title(f"T:{CLASSES[ys_np[i]][:5]}\nP:{CLASSES[preds_np[i]][:5]}", fontsize=7)
plt.suptitle(f"Misclassified samples ({len(wrong)} of {len(ys_np)} wrong)")
plt.tight_layout(); plt.show()

## 9. What did the CNN learn? — filters, feature maps, embeddings

In [ ]:
# First-layer filters of the custom CNN
w = custom.block1[0][0].weight.detach().cpu()
w = (w - w.min()) / (w.max() - w.min())
fig, axes = plt.subplots(4, 16, figsize=(16, 4.2))
for ax, f in zip(axes.ravel(), w):
    ax.imshow(f.permute(1, 2, 0).numpy()); ax.axis("off")
plt.suptitle("First conv layer filters (3x3, RGB) — edge, colour-blob and texture detectors")
plt.tight_layout(); plt.show()

In [ ]:
# Feature maps at increasing depth for one image
img, lbl = test_ds[7]
x = img.unsqueeze(0).to(device)
custom.eval()
with torch.no_grad():
    f1 = custom.block1(x); f2 = custom.block2(f1); f3 = custom.block3(f2)

den = np.array(CIFAR_STD).reshape(3,1,1) * img.numpy() + np.array(CIFAR_MEAN).reshape(3,1,1)
fig = plt.figure(figsize=(16, 5))
ax = fig.add_subplot(3, 13, 1); ax.imshow(den.transpose(1,2,0).clip(0,1))
ax.set_title(f"input\n{CLASSES[lbl]}", fontsize=8); ax.axis("off")
for r, (fm, nm) in enumerate([(f1, "block1 16x16"), (f2, "block2 8x8"), (f3, "block3 4x4")]):
    maps = fm[0].cpu().numpy()
    for k in range(12):
        ax = fig.add_subplot(3, 13, r * 13 + k + 2)
        ax.imshow(maps[k], cmap="viridis"); ax.axis("off")
        if k == 0: ax.set_title(nm, fontsize=8, loc="left")
plt.suptitle("Feature maps become smaller, deeper and more abstract with depth")
plt.tight_layout(); plt.show()

In [ ]:
# t-SNE of the penultimate embedding: does the network separate the classes?
from sklearn.manifold import TSNE

feats, labs = [], []
best.eval()
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(device)
        if best is lenet:
            f = best.features(xb).flatten(1)
        elif best is custom:
            f = best.head[:2](best.block3(best.block2(best.block1(xb))))
        else:
            f = best.head[:2](best.l3(best.l2(best.l1(best.stem(xb)))))
        feats.append(f.cpu()); labs.append(yb)
        if sum(t.shape[0] for t in feats) >= 3000:
            break
feats = torch.cat(feats)[:3000].numpy(); labs = torch.cat(labs)[:3000].numpy()

emb = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(feats)
plt.figure(figsize=(7, 6))
for c in range(10):
    m = labs == c
    plt.scatter(emb[m, 0], emb[m, 1], s=5, alpha=.6, label=CLASSES[c])
plt.legend(markerscale=3, fontsize=8, loc="best")
plt.title(f"t-SNE of {best_name} penultimate features (3000 test images)")
plt.xticks([]); plt.yticks([]); plt.tight_layout(); plt.show()

## 10. Inference demo + saving the model

In [ ]:
torch.save({"model": best.state_dict(), "arch": best_name, "classes": CLASSES},
           "cifar10_best.pth")
print("saved cifar10_best.pth")

idx = np.random.choice(len(test_ds), 8, replace=False)
xb = torch.stack([test_ds[i][0] for i in idx]).to(device)
with torch.no_grad():
    prob = F.softmax(best(xb).float(), dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(17, 5))
for k, i in enumerate(idx):
    axes[0, k].imshow(raw_test.data[i]); axes[0, k].axis("off")
    axes[0, k].set_title(f"true: {CLASSES[test_ds[i][1]]}", fontsize=8)
    top = prob[k].topk(3)
    axes[1, k].barh([CLASSES[j] for j in top.indices][::-1], top.values.numpy()[::-1], color="#4c8fbd")
    axes[1, k].set_xlim(0, 1); axes[1, k].tick_params(labelsize=7)
plt.suptitle("Top-3 predictions"); plt.tight_layout(); plt.show()

## 11. Conclusion

| Model | Params | Typical test acc @15 epochs | Takeaway |
|---|---|---|---|
| LeNet-5 | ~62 K | 62–70 % | Too little capacity and no normalisation; sigmoidal-era design does not scale to natural images. |
| Custom VGG-style CNN | ~1.2 M | 86–90 % | Depth + BN + GAP + dropout is the sweet spot for 32x32 inputs. |
| Compact ResNet | ~2.8 M | 88–92 % | Residual connections let the deeper net train stably and generalise best. |

**Why the custom CNN beats LeNet by a wide margin**
1. **Depth** — 6 conv layers vs 2, so a far larger effective receptive field and a richer hierarchy.
2. **Batch normalisation** — allows a 10x higher learning rate and reduces internal covariate shift.
3. **Global average pooling** — cuts ~1 M FC parameters that LeNet-style heads waste.
4. **Modern regularization** — random crop/flip, dropout, random erasing, weight decay, label smoothing.
5. **Optimizer/schedule** — AdamW with OneCycle beats plain SGD at a fixed small budget.

**Residual errors** cluster on the semantically close pairs (cat/dog, automobile/truck,
deer/horse) — a limitation of 32x32 resolution, not of the architecture.

**To go further:** train 40–60 epochs, add MixUp/CutMix, use a wider ResNet (WRN-28-10) or
fine-tune an ImageNet-pretrained backbone on upscaled 224x224 images for 96 %+.